# 0604 — ETL: cGAN Synthetic Manifold → BigQuery

Bridges the synthetic data generated by the cGAN engine into BigQuery as zero-storage-cost external tables backed by GCS parquet files. This is one of the two bridge notebooks (with `0603`) after which everything downstream reads exclusively from BigQuery, per the project's data-routing canon.

```mermaid
flowchart LR
    A["local parquet\ndf_gan_training_set_v8.parquet"] -->|upload if missing| B["GCS: pienza-streamlit\ndf_gan_training_set_v8.parquet"]
    B --> C["BQ external table\npienza_big.gan_training_set_v8_reference"]

    D["local parquet\n260426_cGAN_manifold_v8.parquet"] -->|upload if missing| E["GCS: pienza-streamlit\n260426_cGAN_manifold_v8.parquet"]
    E --> F["BQ external table\npienza_big.synthetic_manifold_v8"]

    C --> G["row-count verification"]
    F --> G
```


### Phase 1 — Setup

Standard imports, BigQuery client authentication via the notebooks' service account, and the project's plot palette.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from google.cloud import bigquery
from google.oauth2 import service_account

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

# --- Cloud configuration & paths ---
PROJECT_ID = '645009831643'    # BigQuery project ID
DATASET_REAL = 'pienza_mini'   # star schema (real field data)
DATASET_SYNTH = 'pienza_big'   # synthetic cGAN manifold

SA_PATH = "/workspaces/pienza/secrets/service-account.json"
DUMP_DIR = "/workspaces/pienza/data/dumped_files/"

# --- BigQuery client authentication ---
print("Authenticating BigQuery client via service account...")
if os.path.exists(SA_PATH):
    credentials = service_account.Credentials.from_service_account_file(SA_PATH)
    client = bigquery.Client(credentials=credentials, project=PROJECT_ID)
    print(f"BigQuery client ready for project: {PROJECT_ID}")
else:
    print(f"ERROR: service account not found at {SA_PATH}")
    raise FileNotFoundError("Service account missing.")

# --- Plot palette ---
PIENZA_PURPLE, PIENZA_TEAL, PIENZA_GREY, PIENZA_TEXT = '#440154', '#21918c', '#FAFAFA', '#121212'

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': PIENZA_GREY, 'axes.facecolor': PIENZA_GREY, 'text.color': PIENZA_TEXT,
    'axes.titlecolor': PIENZA_PURPLE, 'axes.titleweight': 'bold', 'figure.titlesize': 20
})

print("Plot palette loaded.")
print(f"Ready to operate on {DATASET_REAL} (real) and {DATASET_SYNTH} (synthetic).")


### Phase 2 — Training set: GCS staging + external table

Ensures `df_gan_training_set_v8.parquet` exists in the GCS staging bucket (uploads it if missing), then creates a BigQuery external table pointing at it — no BigQuery storage cost, since the data stays in GCS.

In [ ]:
from google.cloud import bigquery, storage

# --- Paths ---
BUCKET_NAME = 'pienza-streamlit'
LOCAL_PATH = '/workspaces/pienza/data/dumped_files/df_gan_training_set_v8.parquet'
GCS_PATH = 'df_gan_training_set_v8.parquet'

DEST_DATASET = 'pienza_big'
DEST_TABLE = 'gan_training_set_v8_reference'
TABLE_ID = f"{PROJECT_ID}.{DEST_DATASET}.{DEST_TABLE}"


def sync_to_staging():
    print(f"Checking GCS staging: {BUCKET_NAME}...")
    storage_client = storage.Client(credentials=credentials, project=PROJECT_ID)
    bucket = storage_client.bucket(BUCKET_NAME)
    blob = bucket.blob(GCS_PATH)

    if blob.exists():
        print(f"  Found: '{GCS_PATH}' already in GCS.")
    else:
        print(f"  Missing: uploading {os.path.basename(LOCAL_PATH)}...")
        blob.upload_from_filename(LOCAL_PATH)
        print(f"  Uploaded: gs://{BUCKET_NAME}/{GCS_PATH}")


def ensure_dataset_exists():
    dataset_ref = client.dataset(DEST_DATASET)
    try:
        client.get_dataset(dataset_ref)
        print(f"Dataset confirmed: '{DEST_DATASET}'.")
    except Exception:
        print(f"Dataset missing: creating '{DEST_DATASET}'...")
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US"
        client.create_dataset(dataset)
        print("  Dataset created.")


def create_external_table():
    print("Creating BigQuery external table...")

    source_uri = f"gs://{BUCKET_NAME}/{GCS_PATH}"

    external_config = bigquery.ExternalConfig("PARQUET")
    external_config.source_uris = [source_uri]
    external_config.autodetect = True  # BQ infers schema from the parquet file

    table = bigquery.Table(TABLE_ID)
    table.external_data_configuration = external_config

    client.delete_table(TABLE_ID, not_found_ok=True)
    client.create_table(table)

    print(f"Linked: '{DEST_TABLE}' active in '{DEST_DATASET}'.")
    print("BigQuery storage cost: $0.00 (GCS-backed external table).")


print("Starting training set ingest pipeline...")
print("-" * 65)
sync_to_staging()
ensure_dataset_exists()
create_external_table()
print("-" * 65)
print("Training set v8 is online.")


### Phase 3 — Synthetic manifold: GCS staging + external table

Same pattern as Phase 2, applied to the cGAN synthetic manifold (`260426_cGAN_manifold_v8.parquet`), landing as the `synthetic_manifold_v8` external table. Includes a row-count check right after deployment.

In [ ]:
from google.cloud import bigquery, storage

# --- Manifold config ---
MANIFOLD_FILENAME = '260426_cGAN_manifold_v8.parquet'
LOCAL_MANIFOLD_PATH = f'/workspaces/pienza/data/dumped_files/{MANIFOLD_FILENAME}'
GCS_MANIFOLD_PATH = f'{MANIFOLD_FILENAME}'

TABLE_NAME_MANIFOLD = 'synthetic_manifold_v8'
MANIFOLD_TABLE_ID = f"{PROJECT_ID}.{DATASET_SYNTH}.{TABLE_NAME_MANIFOLD}"


def deploy_synthetic_manifold_v8():
    print(f"Connecting synthetic manifold v8 to {DATASET_SYNTH}...")
    print("-" * 65)

    try:
        # A. Staging: ensure the manifold is in GCS
        storage_client = storage.Client(credentials=credentials, project=PROJECT_ID)
        bucket = storage_client.bucket(BUCKET_NAME)
        blob = bucket.blob(GCS_MANIFOLD_PATH)

        if blob.exists():
            print(f"  Found: '{GCS_MANIFOLD_PATH}' already in GCS.")
        else:
            if os.path.exists(LOCAL_MANIFOLD_PATH):
                print("  Missing: uploading manifold v8 from local...")
                blob.upload_from_filename(LOCAL_MANIFOLD_PATH)
                print(f"  Uploaded: gs://{BUCKET_NAME}/{GCS_MANIFOLD_PATH}")
            else:
                raise FileNotFoundError(f"Local file not found: {LOCAL_MANIFOLD_PATH}")

        # B. Forge: configure the BigQuery external table
        print(f"  Creating external table: {TABLE_NAME_MANIFOLD}...")

        uri = f"gs://{BUCKET_NAME}/{GCS_MANIFOLD_PATH}"
        ext_config = bigquery.ExternalConfig("PARQUET")
        ext_config.source_uris = [uri]
        ext_config.autodetect = True

        table = bigquery.Table(MANIFOLD_TABLE_ID)
        table.external_data_configuration = ext_config

        # C. Deploy: replace the table so the pointer stays fresh
        client.delete_table(MANIFOLD_TABLE_ID, not_found_ok=True)
        client.create_table(table)

        print("  Manifold v8 deployed.")
        print(f"  Table: {MANIFOLD_TABLE_ID}")
        print("  BigQuery storage cost: $0.00")

    except Exception as e:
        print(f"  ERROR deploying manifold: {e}")


deploy_synthetic_manifold_v8()

# Quick row-count check
try:
    query = f"SELECT COUNT(*) as total FROM `{MANIFOLD_TABLE_ID}`"
    count = client.query(query).to_dataframe().iloc[0]['total']
    print(f"\nRow count check: {count:,} rows in synthetic manifold v8.")
except Exception as e:
    print(f"Row count check failed: {e}")


### Phase 4 — Verification

Confirms both external tables are live and queryable by counting rows in each.

In [ ]:
for table_name in ['gan_training_set_v8_reference', 'synthetic_manifold_v8']:
    query = f"SELECT COUNT(*) as total FROM `{PROJECT_ID}.{DATASET_SYNTH}.{table_name}`"
    count = client.query(query).to_dataframe().iloc[0]['total']
    print(f"Table {table_name}: {count} rows.")
